# LSTM model for vitals

To configure your system to run this Notebook, from your terminal enter:

```
virtualenv mylstm

source mylstm/bin/activate

python -m pip install numpy scipy matplotlib ipython jupyter pandas sympy nose keras tensorflow scikit-learn

jupyter notebook
```

From Jupyter Notebooks, navigate to your local/clone of this repo and run lstm.ipynb.

In [268]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM

In [269]:
# This suppresses superfluous warnings that pandas throws by default
pd.set_option('mode.chained_assignment', None)

## Source data

This data is sourced from Sentinel and stored locally.

In [270]:
df_vitals = pd.read_csv('vitals_combined_interpolated.csv')
df_vitals.rename(columns = {'reading_date':'timestamp'}, inplace = True)
df_vitals['timestamp'] = pd.to_datetime(df_vitals['timestamp'], errors='coerce')

In [271]:
# Drops batches containing less than 30 observations.
df_recordings_per_patient = df_vitals.groupby(['patient_uuid', 'batch']).count().reset_index()
low_batches = df_recordings_per_patient[(df_recordings_per_patient.HR <= 
                                         30)][['patient_uuid','batch']]
for i in range(len(low_batches)):
    low_patient = low_batches.patient_uuid.iloc[i]
    low_batch = low_batches.batch.iloc[i]
    df_vitals.drop(df_vitals[(df_vitals.patient_uuid == low_patient) & 
                             (df_vitals.batch == low_batch)].index, inplace=True)

In [272]:
df_vitals=df_vitals[(df_vitals.timestamp >= '2018-10-01') & 
                    (df_vitals.timestamp <= '2020-01-14')]

In [273]:
# Systolic Blood Pressure. 
df_vitals_sbp = df_vitals[['timestamp','patient_uuid','SBP']] 

# Heart rate. 
df_vitals_hr = df_vitals[['timestamp','patient_uuid','HR']]

In [274]:
patients = df_vitals.patient_uuid.unique() # to trial run, replace this line with a subset
#patients = patients = df_vitals.patient_uuid.unique()[5:9]
#patients

## LSTM Modeling

This creates an LSTM network for a univariate analysis of vitals passed in as a dataframe with two columns, timestamp and vital sign (eg SBP or HR).

Steps_in defines the look back count (set to 7, here).

Steps_out defines the forward projection count (also set to 7, here).

In [275]:
def split_sequence(sequence, n_steps_in, n_steps_out):
    """Split the sequence for modeling.

    sequence    : pandas dataframe containing the time series variables.
    n_steps_in  : list containing names of the time series variables.    
    n_steps_out : dataframe containing p-values to test causation.
    
    output      : np.array(X) : 
                  np.array(y) : mean absolute error
    """
    X, y = list(), list()
    for i in range(len(sequence)):
        # find the end of this pattern
        end_ix = i + n_steps_in
        out_end_ix = end_ix + n_steps_out
        # check if we are beyond the sequence
        if out_end_ix > len(sequence):
            break
        # gather input and output parts of the pattern
        seq_x, seq_y = sequence[i:end_ix], sequence[end_ix:out_end_ix]
        X.append(seq_x)
        y.append(seq_y)
    return np.array(X), np.array(y)

In [276]:
def lstm_model(dataset):
    """Make a model.

    dataset : pandas dataframe containing the time series variables.
    
    output  : train_result       :
              test_result        :
              testPredict_result :
              yhat_result        :
    """
    # split a univariate sequence into samples
    tsData = dataset.copy(deep=True)
    dataset.drop(columns=['timestamp'], inplace=True)
    scaler = MinMaxScaler(feature_range=(0, 1))
    dataset = scaler.fit_transform(dataset)
    
    train_size = len(dataset) - 7
    test_size = 7 
    train, test = dataset[0:train_size,:], dataset[train_size:len(dataset),:]
    
    train_result = tsData[0:train_size]
    test_result = tsData[train_size:len(tsData)]
    
    # choose a number of time steps
    n_steps_in = 7
    n_steps_out = 7

    # split into samples
    templist = []
    for i in range(len(train)):
        templist.append(train[i][0])

    X, y = split_sequence(templist, n_steps_in, n_steps_out)
    
    templist2 = []
    for j in range(len(test)):
        templist2.append(test[j][0])
    testX, testy = split_sequence(templist2, n_steps_in, n_steps_out)

    # reshape from [samples, timesteps] into [samples, timesteps, features]
    n_features = 1
    X = X.reshape((X.shape[0], X.shape[1], n_features))

    # define model
    model = Sequential()
    model.add(LSTM(100, activation='relu', return_sequences=True, 
                   input_shape=(n_steps_in, n_features)))
    model.add(LSTM(100, activation='relu'))
    model.add(Dense(n_steps_out))
    model.compile(optimizer='adam', loss='mse')

    # fit model
    model.fit(X, y, epochs=50, verbose=0)

    # Evaluate vs. test
    testX = test.reshape((1, n_steps_in, n_features))
    testPredict = model.predict(testX) # no memory
    
    # Make prediction of future
    x_input = dataset[len(dataset)-n_steps_in:len(dataset),:] #must be n_steps_in long

    x_input = x_input.reshape((1, n_steps_in, n_features))
    yhat = model.predict(x_input, verbose=0)
    predictions = scaler.inverse_transform(testPredict)

    predData = {'timestamp':tsData[train_size:len(tsData)]['timestamp'].tolist(), 
        tsData.columns[1]:predictions[0].tolist()} 
    testPredict_result = pd.DataFrame(predData)
    
    ts = tsData['timestamp'][len(tsData)-1:len(tsData)]
    dr = pd.date_range(ts.values[0]+ pd.Timedelta('1 day'), periods=7, freq='D')
    futures = scaler.inverse_transform(yhat)    
    futData = {'timestamp':dr, 
        tsData.columns[1]:futures[0].tolist()} 
    yhat_result = pd.DataFrame(futData)
    
    return train_result, test_result, testPredict_result, yhat_result

In [277]:
def forecast_accuracy(forecast, actual):
    """
     Calculates accuracy by calculating mean absolute percentage error, mean absolute error, 
     and root-mean-squared error.
    
    forecast: forecast dataframe containing the results from a model.
    actual  : test set of a particular patient, with same dates as the forecast dataframe.
    
    output  : mape: mean absolute percentage error
              mae : mean absolute error
              rsme: root mean square error
    """
    #forecast = forecast.values
    
    forecast = forecast.set_index('timestamp')
    actual = actual.set_index('timestamp')
    
    mape = np.mean(np.abs(forecast - actual)/np.abs(actual))  # MAPE
    mae = np.mean(np.abs(forecast - actual))    # MAE
    rmse = np.mean((forecast - actual)**2)**.5  # RMSE
    return mape, mae, rmse

In [278]:
# Runs LSTM model patients and collects data on accuracy.
HR_mape = []
HR_mae = []
HR_rsme = []
SBP_mape = []
SBP_mae = []
SBP_rsme = []
i = 0
data = []
for patient in patients: 
    print('patient ', i, ': ', patient)
    
    df_select_uuid = df_vitals.loc[df_vitals['patient_uuid'] == patient]
    df_SBP = df_select_uuid[['timestamp','SBP']]
    df_HR = df_select_uuid[['timestamp','HR']]    
    
    results_trainS, results_testS, results_forecastS, results_futureS = lstm_model(df_SBP)
    results_trainH, results_testH, results_forecastH, results_futureH = lstm_model(df_HR)
    
    results_train = pd.DataFrame()
    results_train['timestamp'] = results_trainS['timestamp'].values
    results_train['SBP'] = results_trainS['SBP'].values
    results_train['HR'] = results_trainH['HR'].values
    
    results_test = pd.DataFrame()
    results_test['timestamp'] = results_testS['timestamp'].values
    results_test['SBP'] = results_testS['SBP'].values
    results_test['HR'] = results_testH['HR'].values

    results_forecast = pd.DataFrame()
    results_forecast['timestamp'] = results_forecastS['timestamp'].values
    results_forecast['SBP'] = results_forecastS['SBP'].values
    results_forecast['HR'] = results_forecastH['HR'].values
    
    results_future = pd.DataFrame()
    results_future['timestamp'] = results_futureS['SBP'].values
    results_future['SBP'] = results_futureS['SBP'].values
    results_future['HR'] = results_futureH['HR'].values
    
    i = i + 1    
        
    accuracy_indicators = forecast_accuracy(results_forecast, results_test)
    HR_mape.append(accuracy_indicators[0][0])
    HR_mae.append(accuracy_indicators[1][0])
    HR_rsme.append(accuracy_indicators[2][0])
    SBP_mape.append(accuracy_indicators[0][1])
    SBP_mae.append(accuracy_indicators[1][1])
    SBP_rsme.append(accuracy_indicators[2][1])
    
    df_train = results_train.set_index('timestamp')
    df_test  = results_test.set_index('timestamp')
    df_forecast = results_forecast.set_index('timestamp')
    df_future = results_future.set_index('timestamp')
    
    flag_row = compare_flags(df_train, df_test, df_forecast)
    flag_row = [patient] + flag_row
    data.append(flag_row)

patient  0 :  1127912
patient  1 :  1164638
patient  2 :  1166047
patient  3 :  1193821
patient  4 :  1231804
patient  5 :  1519674
patient  6 :  1523219
patient  7 :  2075791
patient  8 :  2096100
patient  9 :  2104097
patient  10 :  2132729
patient  11 :  2168037
patient  12 :  2172350
patient  13 :  2469101
patient  14 :  2714491
patient  15 :  2781252
patient  16 :  2793778
patient  17 :  2802493
patient  18 :  2823949
patient  19 :  3005945
patient  20 :  3224753
patient  21 :  3371769
patient  22 :  3415879
patient  23 :  3429852
patient  24 :  3469162
patient  25 :  3501786
patient  26 :  3581867
patient  27 :  3725893
patient  28 :  3832317
patient  29 :  3862758
patient  30 :  3867047
patient  31 :  3958757
patient  32 :  3959150
patient  33 :  3981039
patient  34 :  4026680
patient  35 :  4167424
patient  36 :  4182532
patient  37 :  4650253
patient  38 :  4687330
patient  39 :  4773302
patient  40 :  5190247
patient  41 :  5194444
patient  42 :  5363765
patient  43 :  549077

patient  336 :  36279955
patient  337 :  36291435
patient  338 :  36309033
patient  339 :  36383824
patient  340 :  36398581
patient  341 :  36641898
patient  342 :  36786019
patient  343 :  36824109
patient  344 :  36893944
patient  345 :  36950130
patient  346 :  37200595
patient  347 :  37460481
patient  348 :  37493202
patient  349 :  37554403
patient  350 :  37591865
patient  351 :  37697105
patient  352 :  37971276
patient  353 :  38075552
patient  354 :  38077287
patient  355 :  38208759
patient  356 :  38438063
patient  357 :  38528370
patient  358 :  38664428
patient  359 :  38982909
patient  360 :  39012657
patient  361 :  39021941
patient  362 :  39257949
patient  363 :  39531899
patient  364 :  39742402
patient  365 :  40474386
patient  366 :  40519999
patient  367 :  40858257
patient  368 :  40863796
patient  369 :  41008900
patient  370 :  41166027
patient  371 :  41185349
patient  372 :  41265500
patient  373 :  41268809
patient  374 :  41283672
patient  375 :  41385745


patient  664 :  74666273
patient  665 :  74953225
patient  666 :  74959813
patient  667 :  74969507
patient  668 :  74995309
patient  669 :  75037532
patient  670 :  75125218
patient  671 :  75391777
patient  672 :  75667697
patient  673 :  75710605
patient  674 :  76145603
patient  675 :  76564643
patient  676 :  76568159
patient  677 :  76607038
patient  678 :  76689506
patient  679 :  76752562
patient  680 :  76859127
patient  681 :  76923331
patient  682 :  76989480
patient  683 :  77041013
patient  684 :  77064905
patient  685 :  77071946
patient  686 :  77201659
patient  687 :  77283126
patient  688 :  77294315
patient  689 :  77462882
patient  690 :  77630830
patient  691 :  77655734
patient  692 :  77659383
patient  693 :  77715731
patient  694 :  77762729
patient  695 :  77818682
patient  696 :  78063006
patient  697 :  78409457
patient  698 :  78744350
patient  699 :  78859592
patient  700 :  79071662
patient  701 :  79453504
patient  702 :  79475980
patient  703 :  79546710


In [279]:
# Creates and populates dataframe containing data on accuracy of the models for each patient.
df_model = pd.DataFrame(columns = ['patient_uuid', 'HR_mape', 'HR_mae', 'HR_rsme', 'SBP_mape', 'SBP_mae', 'SBP_rsme'])
df_model.patient_uuid = patients #patients
df_model.HR_mape = HR_mape
df_model.HR_mae = HR_mae
df_model.HR_rsme = HR_rsme
df_model.SBP_mape = SBP_mape
df_model.SBP_mae = SBP_mae
df_model.SBP_rsme = SBP_rsme

#df_model.head()

In [280]:
# Creates the raised flag dataframe.
flag_columns = ['patient_uuid', 'HR_error', 'SBP_error', 'algo_accuracy', 'actual_rising_HR', 'actual_lowering_HR',
                'actual_rising_SBP', 'actual_min_HR', 'actual_max_HR', 'actual_min_SBP', 'actual_max_SBP', 'actual_high_BP',
                'pred_rising_HR', 'pred_lowering_HR', 'pred_rising_SBP', 'pred_min_HR', 'pred_max_HR', 'pred_min_SBP',
                'pred_max_SBP', 'pred_high_BP']

df_flags = pd.DataFrame(data, columns=flag_columns)
df_flags = df_flags[['patient_uuid', 'HR_error', 'SBP_error', 'algo_accuracy', 'pred_rising_HR', 'actual_rising_HR',
                     'pred_lowering_HR', 'actual_lowering_HR', 'pred_rising_SBP', 'actual_rising_SBP', 'pred_high_BP',
                     'actual_high_BP', 'pred_min_HR', 'actual_min_HR', 'pred_max_HR', 'actual_max_HR', 'pred_min_SBP', 
                     'actual_min_SBP', 'pred_max_SBP', 'actual_max_SBP']]

In [281]:
df_flags

,patient_uuid,HR_error,SBP_error,algo_accuracy,pred_rising_HR,actual_rising_HR,pred_lowering_HR,actual_lowering_HR,pred_rising_SBP,actual_rising_SBP,pred_high_BP,actual_high_BP,pred_min_HR,actual_min_HR,pred_max_HR,actual_max_HR,pred_min_SBP,actual_min_SBP,pred_max_SBP,actual_max_SBP
0,1127912,0.339624,0.245094,0.625,True,False,False,False,False,False,False,False,0,0,7,1,7,0,0,0
1,1164638,0.304522,0.209839,0.625,True,False,False,False,False,False,False,False,0,0,7,0,7,0,0,0
2,1166047,0.429736,0.285840,0.625,True,False,False,False,False,False,False,False,0,0,7,0,7,0,0,0
3,1193821,0.557391,0.308716,0.625,True,False,False,False,False,False,False,False,0,0,7,0,7,0,0,0
4,1231804,0.250621,0.231084,0.500,True,False,False,False,False,True,False,False,0,0,7,0,16,9,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
878,98772423,0.936187,0.398311,0.625,True,False,False,False,False,False,False,False,0,0,7,0,7,0,0,0
879,98999464,0.546218,0.306252,0.625,True,False,False,False,False,False,False,False,0,0,7,0,7,0,0,0
880,99535439,0.679767,0.331598,0.625,True,False,False,False,False,False,False,False,0,0,7,0,7,0,0,0
881,99683668,0.482936,0.290217,0.625,True,False,False,False,False,False,False,False,0,0,7,0,7,0,0,0


In [282]:
def forecast_accuracy(forecast, actual):
    """
     Calculates accuracy by calculating mean absolute percentage error, mean absolute error, 
     and root-mean-squared error.
    
    forecast: forecast dataframe containing the results from a model.
    actual  : test set of a particular patient, with same dates as the forecast dataframe.
    
    output  : mape: mean absolute percentage error
              mae : mean absolute error
              rsme: root mean square error
    """
    #forecast = forecast.values
    
    forecast = forecast.set_index('timestamp')
    actual = actual.set_index('timestamp')
    
    mape = np.mean(np.abs(forecast - actual)/np.abs(actual))  # MAPE
    mae = np.mean(np.abs(forecast - actual))    # MAE
    rmse = np.mean((forecast - actual)**2)**.5  # RMSE
    return mape, mae, rmse

In [283]:
# create a dataframe of average metrics
df_lstm_metrics = pd.DataFrame((
    df_model[['HR_mape', 'HR_mae', 'HR_rsme', 'SBP_mape','SBP_mae','SBP_rsme']].mean()), 
    columns=['LSTM'])
df_lstm_metrics

,LSTM
HR_mape,0.035034
HR_mae,4.438237
HR_rsme,5.509894
SBP_mape,0.035014
SBP_mae,2.573227
SBP_rsme,3.208053


In [284]:
# save model results to file
df_model.to_csv('df_model_lstm.csv')
df_flags.to_csv('df_flags_lstm.csv')
df_lstm_metrics.to_csv('df_lstm_metrics.csv')

## Flag setting

Push results through the flag setting model as documented by Agustin in VAR.

In [285]:
def compare_flags(df_train, df_test, df_forecast):
    """
    Compares the results of the flag-raising algorithm.
    
    df_train   : training set of the timeseries for a particular patient.
    df_test    : test set of the timeseries for a particular patient.
    df_forecast: forecast of the timeseries for a particular patient.
    
    output     : list containing HR_error, SBP_error, algorithm accuracy, 
                 predicted flags and actual flags.
    """
    flag_comparison = []
    current = df_train[['HR','SBP']][-23:].copy()
    future = df_test[['HR','SBP']].copy()
    forecast = df_forecast.copy()
    forecast.columns = ['HR', 'SBP']
    actual = pd.concat([current, future])
    pred = pd.concat([current, forecast])
    
    actual_flags = flags(actual)
    pred_flags = flags(pred)
    
    HR_error = np.abs(pred_flags[0][0] - actual_flags[0][0]) / actual_flags[0][0]
    SBP_error = np.abs(pred_flags[0][1] - actual_flags[0][1]) / actual_flags[0][1]

    for i in range(len(actual_flags)):
        flag_comparison.append(actual_flags[i] == pred_flags[i])
    
    algo_accuracy = np.sum(flag_comparison[1:]) / 8
    
    result = [HR_error, SBP_error, algo_accuracy] + actual_flags[1:] + pred_flags[1:]
    
    return result

In [286]:
def flags(df, report=False):
    """
    Flag-raising algorithm based on health deterioration rules.
    
    df    : 30-day timeseries pandas dataframe.
    report: True if written report requested. False if status of each flag is wanted.
    
    output: returns a list containing the average value of HR and SBP during the last ten days of the time series,
            and the different flags that are raised based on the conditions above.
    """
    rising_HR = False
    lowering_HR = False
    rising_SBP = False
    high_BP = False
    
    first_ten = np.mean(df[:10])
    last_ten = np.mean(df[-10:])

    deltas = (last_ten - first_ten)
    
    if deltas[0] >= 10:
        rising_HR = True
        flag_message_1 = 'Detected rising heart rate. \n'
    elif deltas[0] <= -10:
        lowering_HR = True
        flag_message_1 = 'Detected decreasing heart rate. \n'
    else:
        flag_message_1 = 'No changes in heart rate detected. \n'
        
    if deltas[1] >= 5:
        rising_SBP = True
        flag_message_2 = 'Detected rising blood pressure (SBP). \n'
    else:
        flag_message_2 = 'No changes in blood pressure detected. \n'

    min_HR = np.sum(df.HR < 55)
    max_HR = np.sum(df.HR > 100)

    min_SBP = np.sum(df.SBP < 90)
    max_SBP = np.sum(df.SBP > 180)
    
    flag_message_3 = 'Heart rate lower than 55 bpm detected ' + str(min_HR) + ' times. \n'
    flag_message_4 = 'Heart rate greater than 100 bpm detected ' + str(max_HR) + ' times. \n'
    flag_message_5 = 'Systolic blood pressure (SBP) lower than 90 bpm detected ' + str(min_SBP) + ' times. \n'
    flag_message_6 = 'Systolic blood pressure (SBP) greater than 180 bpm detected ' + str(max_SBP) + ' times. \n'
    flag_message_7 = ''
    
    high_SBP = (df.SBP > 140)
    
    if np.sum(high_SBP[-7:]) > 3:
        high_BP = True
        flag_message_7 = 'High blood pressure detected.'
    if report == True:
        if high_BP == True:
            message = flag_message_1 + flag_message_2 + flag_message_3 + flag_message_4 + flag_message_5 + flag_message_6 + flag_message_7 
        else:
            message = flag_message_1 + flag_message_2 + flag_message_3 + flag_message_4 + flag_message_5 + flag_message_6
        print(message)
    else:
        return [last_ten, rising_HR, lowering_HR, rising_SBP, min_HR, max_HR, min_SBP, max_SBP, high_BP]

In [287]:
def accuracy_f1(true_positive, true_negative, false_positive, false_negative):
    """
    Calculates accuracy and F1-score for a particular feature.
    true_positive : number of true positives.
    true_negative : number of true negatives.
    false_positive: number of false positives.
    false_negative: number of false negatives.
    output:    acc: accuracy of the model on the feature.
          f1_score: F1-score of the model on the feature.
    """
    acc = (true_positive + true_negative) / (true_positive + false_positive + false_negative + true_negative)
    prec = true_positive / (true_positive + false_positive)
    rec = true_positive / (true_positive + false_negative)
    f1_score = 2 * (rec * prec) / (rec + prec)
    if (rec == 0) & (prec == 0):
        f1_score = 0
    return acc, f1_score

In [288]:
#rising hr
tp_rising_HR = np.sum((df_flags.pred_rising_HR == True) & (df_flags.actual_rising_HR == True))
tn_rising_HR = np.sum((df_flags.pred_rising_HR == False) & (df_flags.actual_rising_HR == False))
fp_rising_HR = np.sum((df_flags.pred_rising_HR == True) & (df_flags.actual_rising_HR == False))
fn_rising_HR = np.sum((df_flags.pred_rising_HR == False) & (df_flags.actual_rising_HR == True))
#lowering hr
tp_lowering_HR = np.sum((df_flags.pred_lowering_HR == True) & (df_flags.actual_lowering_HR == True))
tn_lowering_HR = np.sum((df_flags.pred_lowering_HR == False) & (df_flags.actual_lowering_HR == False))
fp_lowering_HR = np.sum((df_flags.pred_lowering_HR == True) & (df_flags.actual_lowering_HR == False))
fn_lowering_HR = np.sum((df_flags.pred_lowering_HR == False) & (df_flags.actual_lowering_HR == True))
# rising sbp
tp_rising_SBP = np.sum((df_flags.pred_rising_SBP == True) & (df_flags.actual_rising_SBP == True))
tn_rising_SBP = np.sum((df_flags.pred_rising_SBP == False) & (df_flags.actual_rising_SBP == False))
fp_rising_SBP = np.sum((df_flags.pred_rising_SBP == True) & (df_flags.actual_rising_SBP == False))
fn_rising_SBP = np.sum((df_flags.pred_rising_SBP == False) & (df_flags.actual_rising_SBP == True))
# high_bp
tp_high_BP = np.sum((df_flags.pred_high_BP == True) & (df_flags.actual_high_BP == True))
tn_high_BP = np.sum((df_flags.pred_high_BP == False) & (df_flags.actual_high_BP == False))
fp_high_BP = np.sum((df_flags.pred_high_BP == True) & (df_flags.actual_high_BP == False))
fn_high_BP = np.sum((df_flags.pred_high_BP == False) & (df_flags.actual_high_BP == True))
# metrics for binary flags
acc_rising_HR, f1_rising_HR = accuracy_f1(tp_rising_HR, tn_rising_HR, fp_rising_HR, fn_rising_HR)
acc_lowering_HR, f1_lowering_HR = accuracy_f1(tp_lowering_HR, tn_lowering_HR, fp_lowering_HR, fn_lowering_HR)
acc_rising_SBP, f1_rising_SBP = accuracy_f1(tp_rising_SBP, tn_rising_SBP, fp_rising_SBP, fn_rising_SBP)
acc_high_BP, f1_high_BP = accuracy_f1(tp_high_BP, tn_high_BP, fp_high_BP, fn_high_BP)
# mae for numerical flags
mae_min_HR = np.abs(df_flags.pred_min_HR - df_flags.actual_min_HR).mean()
mae_max_HR = np.abs(df_flags.pred_max_HR - df_flags.actual_max_HR).mean()
mae_min_SBP = np.abs(df_flags.pred_min_SBP - df_flags.actual_min_SBP).mean()
mae_max_SBP = np.abs(df_flags.pred_max_SBP - df_flags.actual_max_SBP).mean()
algo_metrics = [acc_rising_HR, f1_rising_HR, acc_lowering_HR, f1_lowering_HR, acc_rising_SBP, f1_rising_SBP,
                acc_high_BP, f1_high_BP, mae_min_HR, mae_max_HR, mae_min_SBP, mae_max_SBP]
algo_metrics_index = ['acc_rising_HR', 'f1_rising_HR', 'acc_lowering_HR', 'f1_lowering_HR', 'acc_rising_SBP',
                      'f1_rising_SBP','acc_high_BP', 'f1_high_BP', 'mae_min_HR', 'mae_max_HR', 'mae_min_SBP',
                      'mae_max_SBP']
algo_metrics

/Users/pking/lstm/lib/python3.7/site-packages/ipykernel_launcher.py:14: RuntimeWarning: invalid value encountered in double_scalars
  


[0.01698754246885617,
 0.011389521640091117,
 0.9818799546998868,
 nan,
 0.8912797281993206,
 0.02040816326530612,
 0.7746319365798414,
 nan,
 0.2650056625141563,
 6.805209513023782,
 6.498301245753114,
 0.022650056625141562]

In [289]:
df_flags

,patient_uuid,HR_error,SBP_error,algo_accuracy,pred_rising_HR,actual_rising_HR,pred_lowering_HR,actual_lowering_HR,pred_rising_SBP,actual_rising_SBP,pred_high_BP,actual_high_BP,pred_min_HR,actual_min_HR,pred_max_HR,actual_max_HR,pred_min_SBP,actual_min_SBP,pred_max_SBP,actual_max_SBP
0,1127912,0.339624,0.245094,0.625,True,False,False,False,False,False,False,False,0,0,7,1,7,0,0,0
1,1164638,0.304522,0.209839,0.625,True,False,False,False,False,False,False,False,0,0,7,0,7,0,0,0
2,1166047,0.429736,0.285840,0.625,True,False,False,False,False,False,False,False,0,0,7,0,7,0,0,0
3,1193821,0.557391,0.308716,0.625,True,False,False,False,False,False,False,False,0,0,7,0,7,0,0,0
4,1231804,0.250621,0.231084,0.500,True,False,False,False,False,True,False,False,0,0,7,0,16,9,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
878,98772423,0.936187,0.398311,0.625,True,False,False,False,False,False,False,False,0,0,7,0,7,0,0,0
879,98999464,0.546218,0.306252,0.625,True,False,False,False,False,False,False,False,0,0,7,0,7,0,0,0
880,99535439,0.679767,0.331598,0.625,True,False,False,False,False,False,False,False,0,0,7,0,7,0,0,0
881,99683668,0.482936,0.290217,0.625,True,False,False,False,False,False,False,False,0,0,7,0,7,0,0,0
